# Lab 2A – Klasyczna detekcja znaków drogowych i sygnalizatorów

## Zadanie 1: wczytanie danych i wizualizacja ramek referencyjnych

Docelowe klasy: `traffic_sign`, `traffic_light_red`, `traffic_light_yellow`, `traffic_light_green`, `traffic_light_unknown`.
W BDD100K sygnalizator z `trafficLightColor == "none"` traktujemy jako `traffic_light_unknown`.

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from PIL import Image

RANDOM_STATE = 7777
random.seed(RANDOM_STATE)

DATA_DIR = Path(".")
IMG_DIR = DATA_DIR / "bdd100k_images_100k" / "100k"
LBL_DIR = DATA_DIR / "bdd100k_labels" / "100k"
SPLITS = ["train", "val", "test"]

TARGET_CLASSES = [
    "traffic_sign",
    "traffic_light_red",
    "traffic_light_yellow",
    "traffic_light_green",
    "traffic_light_unknown",
]

CLASS_COLORS = {
    "traffic_sign": "#00d5ff",
    "traffic_light_red": "#ff2020",
    "traffic_light_yellow": "#ffd400",
    "traffic_light_green": "#20e040",
    "traffic_light_unknown": "#ff40ff",
}

### Wczytywanie etykiet

Każdy obraz ma osobny plik JSON (`frames[0].objects`). Zostawiamy tylko obiekty z `box2d`
należące do kategorii `traffic sign` / `traffic light` i mapujemy je na docelowe klasy.

In [ ]:
def map_class(obj):
    """Mapuje obiekt BDD100K na klasę docelową lub None, jeśli nas nie interesuje."""
    cat = obj.get("category")
    if cat == "traffic sign":
        return "traffic_sign"
    if cat == "traffic light":
        color = obj.get("attributes", {}).get("trafficLightColor", "none")
        if color in ("red", "yellow", "green"):
            return f"traffic_light_{color}"
        return "traffic_light_unknown"
    return None


def load_annotation(label_path):
    """Zwraca słownik z nazwą obrazu, atrybutami sceny i listą ramek docelowych klas."""
    with open(label_path) as f:
        data = json.load(f)

    boxes = []
    for frame in data.get("frames", []):
        for obj in frame.get("objects", []):
            if "box2d" not in obj:
                continue
            cls = map_class(obj)
            if cls is None:
                continue
            b = obj["box2d"]
            attrs = obj.get("attributes", {})
            boxes.append({
                "cls": cls,
                "x1": b["x1"], "y1": b["y1"], "x2": b["x2"], "y2": b["y2"],
                "occluded": attrs.get("occluded", False),
                "truncated": attrs.get("truncated", False),
            })

    return {
        "name": data["name"],
        "attributes": data.get("attributes", {}),
        "boxes": boxes,
    }


def load_split(split, limit=None):
    """Wczytuje adnotacje z danego splitu (opcjonalnie tylko `limit` pierwszych plików)."""
    files = sorted((LBL_DIR / split).glob("*.json"))
    if limit is not None:
        files = files[:limit]
    return [load_annotation(p) for p in files]


def image_path(ann, split):
    return IMG_DIR / split / f"{ann['name']}.jpg"

In [ ]:
# Do wizualizacji wystarczy podzbiór – pełny train (70k plików) wczytuje się dłużej.
N_LOAD = 5000
anns = {split: load_split(split, limit=N_LOAD) for split in SPLITS}

for split, a in anns.items():
    n_boxes = sum(len(x["boxes"]) for x in a)
    print(f"{split:5s}: {len(a)} obrazów, {n_boxes} ramek docelowych klas")

### Rysowanie ramek

In [ ]:
def draw_boxes(ax, ann, split, show_labels=True, linewidth=1.5):
    """Rysuje obraz i jego ramki referencyjne na podanej osi."""
    img = Image.open(image_path(ann, split)).convert("RGB")
    ax.imshow(img)

    for b in ann["boxes"]:
        color = CLASS_COLORS[b["cls"]]
        w, h = b["x2"] - b["x1"], b["y2"] - b["y1"]
        # ramki zasłonięte / ucięte rysujemy przerywaną linią
        ls = "--" if (b["occluded"] or b["truncated"]) else "-"
        ax.add_patch(patches.Rectangle(
            (b["x1"], b["y1"]), w, h,
            fill=False, edgecolor=color, linewidth=linewidth, linestyle=ls,
        ))
        if show_labels:
            label = b["cls"].replace("traffic_light_", "TL ").replace("traffic_sign", "TS")
            ax.text(b["x1"], b["y1"] - 3, label, color="black", fontsize=6,
                    bbox=dict(facecolor=color, edgecolor="none", pad=1, alpha=0.85))

    attrs = ann["attributes"]
    title = f"{ann['name']}\n{attrs.get('timeofday', '?')} | {attrs.get('weather', '?')} | {attrs.get('scene', '?')}"
    ax.set_title(title, fontsize=8)
    ax.axis("off")


def legend_handles():
    handles = [Line2D([0], [0], color=c, lw=2, label=k) for k, c in CLASS_COLORS.items()]
    handles.append(Line2D([0], [0], color="gray", lw=2, ls="--", label="occluded / truncated"))
    return handles


def show_grid(samples, split, ncols=3, figsize_per=(6, 3.6), suptitle=None, **kw):
    """Wyświetla siatkę obrazów z ramkami."""
    if not samples:
        print("Brak obrazów spełniających kryteria.")
        return
    nrows = (len(samples) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per[0] * ncols, figsize_per[1] * nrows))
    axes = list(axes.flat) if hasattr(axes, "flat") else [axes]
    for ax, ann in zip(axes, samples):
        draw_boxes(ax, ann, split, **kw)
    for ax in axes[len(samples):]:
        ax.axis("off")
    fig.legend(handles=legend_handles(), loc="lower center", ncol=6, fontsize=9, frameon=False)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    fig.tight_layout(rect=(0, 0.04, 1, 0.97))
    plt.show()


def select(split, n=6, min_boxes=1, **attr_filter):
    """Losuje `n` obrazów z co najmniej `min_boxes` ramkami, spełniających filtry atrybutów sceny,
    np. select("train", timeofday="night", weather="rainy")."""
    pool = [
        a for a in anns[split]
        if len(a["boxes"]) >= min_boxes
        and all(a["attributes"].get(k) == v for k, v in attr_filter.items())
    ]
    return random.sample(pool, min(n, len(pool)))

### Losowe obrazy (warunki dzienne)

In [ ]:
show_grid(select("train", n=6, min_boxes=3, timeofday="daytime", weather="clear"), "train",
          suptitle="Train – dzień, bezchmurnie")

### Trudniejsze warunki: noc, deszcz, śnieg, mgła, świt/zmierzch

In [ ]:
hard_conditions = [
    ("Noc", dict(timeofday="night")),
    ("Deszcz", dict(weather="rainy")),
    ("Śnieg", dict(weather="snowy")),
    ("Mgła", dict(weather="foggy")),
    ("Świt / zmierzch", dict(timeofday="dawn/dusk")),
    ("Noc + deszcz", dict(timeofday="night", weather="rainy")),
]

for title, flt in hard_conditions:
    show_grid(select("train", n=3, min_boxes=2, **flt), "train", suptitle=f"Train – {title}")

### Każda klasa sygnalizatora – przykłady

In [ ]:
for cls in TARGET_CLASSES[1:]:
    pool = [a for a in anns["train"] if any(b["cls"] == cls for b in a["boxes"])]
    show_grid(random.sample(pool, min(3, len(pool))), "train", suptitle=f"Obrazy zawierające {cls}")

### Przypadki szczególne: bardzo małe ramki i ramki ucięte / zasłonięte

In [ ]:
def box_area(b):
    return (b["x2"] - b["x1"]) * (b["y2"] - b["y1"])

tiny = [a for a in anns["train"] if any(box_area(b) < 100 for b in a["boxes"])]
show_grid(random.sample(tiny, min(3, len(tiny))), "train", suptitle="Obrazy z ramkami < 100 px²")

occl = [a for a in anns["train"] if sum(b["occluded"] or b["truncated"] for b in a["boxes"]) >= 2]
show_grid(random.sample(occl, min(3, len(occl))), "train", suptitle="Obrazy z ramkami occluded / truncated")

### Zbliżenie na ramki (crop)

Ramki znaków i świateł są często małe, więc warto zobaczyć same wycinki.

In [ ]:
def show_crops(split, cls, n=16, pad=4, ncols=8):
    crops = []
    pool = [a for a in anns[split] if any(b["cls"] == cls for b in a["boxes"])]
    random.shuffle(pool)
    for ann in pool:
        img = Image.open(image_path(ann, split)).convert("RGB")
        for b in ann["boxes"]:
            if b["cls"] == cls:
                crops.append(img.crop((b["x1"] - pad, b["y1"] - pad, b["x2"] + pad, b["y2"] + pad)))
        if len(crops) >= n:
            break
    crops = crops[:n]
    nrows = (len(crops) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.6 * ncols, 1.8 * nrows))
    for ax in axes.flat:
        ax.axis("off")
    for ax, c in zip(axes.flat, crops):
        ax.imshow(c)
        ax.set_title(f"{c.size[0]}×{c.size[1]}", fontsize=7)
    fig.suptitle(cls)
    fig.tight_layout()
    plt.show()

for cls in TARGET_CLASSES:
    show_crops("train", cls)

### Podgląd konkretnego obrazu

Wpisz nazwę pliku (bez `.jpg`), żeby obejrzeć wybrany obraz w dużej rozdzielczości.

In [ ]:
def show_one(name, split="train"):
    ann = load_annotation(LBL_DIR / split / f"{name}.json")
    fig, ax = plt.subplots(figsize=(14, 8))
    draw_boxes(ax, ann, split, linewidth=2)
    ax.legend(handles=legend_handles(), loc="upper right", fontsize=8)
    plt.show()
    print(Counter(b["cls"] for b in ann["boxes"]))

show_one("0000f77c-6257be58", "train")

## Statystyki rozmiarów ramek – cały zbiór treningowy

Liczymy rozmiary wszystkich ramek docelowych klas w `train` (70 000 plików).
Wyniki posłużą do doboru rozmiaru okna HOG, minimalnego rozmiaru kandydata i filtrów proporcji.

In [ ]:
import numpy as np
import pandas as pd

train_all = load_split("train")  # pełny train, ~1 min

box_df = pd.DataFrame([
    {"image": a["name"], "cls": b["cls"], "x1": b["x1"], "y1": b["y1"], "x2": b["x2"], "y2": b["y2"],
     "occluded": b["occluded"], "truncated": b["truncated"],
     "timeofday": a["attributes"].get("timeofday")}
    for a in train_all for b in a["boxes"]
])
box_df["w"] = box_df["x2"] - box_df["x1"]
box_df["h"] = box_df["y2"] - box_df["y1"]
box_df["area"] = box_df["w"] * box_df["h"]
box_df["min_side"] = box_df[["w", "h"]].min(axis=1)
box_df["aspect"] = box_df["w"] / box_df["h"].clip(lower=1e-6)
box_df["cy_rel"] = (box_df["y1"] + box_df["y2"]) / 2 / 720
# grupa: znak / sygnalizator (bez podziału na kolor)
box_df["group"] = np.where(box_df["cls"] == "traffic_sign", "traffic_sign", "traffic_light")

print(f"{len(train_all)} obrazów, {len(box_df)} ramek")
box_df.groupby("cls").size()

In [ ]:
def size_summary(df, by):
    g = df.groupby(by)
    return pd.DataFrame({
        "liczba": g.size(),
        "śr. w": g["w"].mean(), "śr. h": g["h"].mean(),
        "med. w": g["w"].median(), "med. h": g["h"].median(),
        "śr. pole": g["area"].mean(), "med. pole": g["area"].median(),
        "med. w/h": g["aspect"].median(),
        "% min bok<10": g["min_side"].apply(lambda s: (s < 10).mean() * 100),
        "% min bok<16": g["min_side"].apply(lambda s: (s < 16).mean() * 100),
        "% min bok<32": g["min_side"].apply(lambda s: (s < 32).mean() * 100),
    }).round(1)

display(size_summary(box_df, "group"))
display(size_summary(box_df, "cls"))

In [ ]:
# percentyle szerokości i wysokości
pct = [10, 25, 50, 75, 90, 99]
for grp, d in box_df.groupby("group"):
    print(grp)
    print(pd.DataFrame({"w": np.percentile(d["w"], pct), "h": np.percentile(d["h"], pct)},
                       index=[f"p{p}" for p in pct]).round(1).T, "\n")

### Rozkłady rozmiarów i proporcji

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
colors = {"traffic_sign": CLASS_COLORS["traffic_sign"], "traffic_light": CLASS_COLORS["traffic_light_red"]}
bins = np.logspace(0, np.log10(500), 60)

for grp, d in box_df.groupby("group"):
    axes[0].hist(np.sqrt(d["area"]), bins=bins, alpha=0.6, label=grp, color=colors[grp])
    axes[1].hist(np.log2(d["aspect"].clip(1/16, 16)), bins=60, alpha=0.6, label=grp, color=colors[grp])
    axes[2].hist(d["cy_rel"], bins=50, alpha=0.6, label=grp, color=colors[grp], orientation="horizontal")

axes[0].set_xscale("log"); axes[0].set_xlabel("√pole [px] (skala log)"); axes[0].set_title("Rozmiar ramki")
for t in (10, 32):
    axes[0].axvline(t, color="gray", ls="--", lw=1)
axes[1].set_xlabel("log2(w / h)   (<0 pionowe, >0 poziome)"); axes[1].set_title("Proporcje ramki")
axes[2].invert_yaxis(); axes[2].set_ylabel("położenie środka w pionie (0 = góra kadru)"); axes[2].set_title("Położenie w pionie")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# szerokość vs wysokość (gęstość) – osobno dla znaków i sygnalizatorów
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (grp, d) in zip(axes, box_df.groupby("group")):
    hb = ax.hexbin(d["w"], d["h"], gridsize=60, extent=(0, 120, 0, 120), bins="log", cmap="viridis", mincnt=1)
    ax.plot([0, 120], [0, 120], "w--", lw=1)
    ax.set_xlabel("w [px]"); ax.set_ylabel("h [px]"); ax.set_title(grp)
    fig.colorbar(hb, ax=ax, label="log(liczba)")
plt.tight_layout()
plt.show()

In [ ]:
# rozmiar a pora dnia
display(box_df.groupby(["group", "timeofday"])["area"].agg(["size", "mean", "median"]).round(0))

# rozmiar a położenie w pionie: obiekty bliżej horyzontu są mniejsze
box_df["cy_bin"] = pd.cut(box_df["cy_rel"], bins=np.linspace(0, 1, 11))
display(box_df.pivot_table(index="cy_bin", columns="group", values="min_side", aggfunc="median", observed=True).round(1))

In [ ]:
# ramki zdegenerowane / podejrzane – artefakty adnotacji
degenerate = box_df[(box_df["w"] <= 1) | (box_df["h"] <= 1)]
huge = box_df[box_df["area"] > 0.1 * 1280 * 720]
print(f"Ramki zdegenerowane (bok <= 1 px): {len(degenerate)}")
print(degenerate.groupby("cls").size(), "\n")
print(f"Ramki ogromne (> 10% kadru): {len(huge)}")
print(huge.groupby("cls").size())

**Wnioski:**
- **Rozmiar:** rozkład jest prawoskośny – mediana (znak ~24×19 px, sygnalizator ~14×20 px) jest wyraźnie mniejsza od średniej (32×25 px i 16×25 px). Typowy obiekt jest mały, a średnią zawyżają nieliczne duże ramki.
- **Okno HOG:** 32×32 dla znaków i ~16×32 dla sygnalizatorów odpowiada typowym rozmiarom (83% znaków ma krótszy bok < 32 px, więc większość będzie lekko powiększana).
- **Małe obiekty:** odrzucenie ramek o krótszym boku < 10 px z góry obniża osiągalny recall o ~16% dla znaków i ~30% dla sygnalizatorów.
- **Proporcje:** sygnalizatory są pionowe (w/h ≈ 0,6). Rozkład proporcji znaków jest **dwumodalny** – znaki prawie kwadratowe (w/h ≈ 1) oraz szerokie tablice (w/h ≈ 2) – co uzasadnia dwa okna / dwa klasyfikatory dla znaków albo dołożenie proporcji jako cechy.
- **Położenie:** środki ramek leżą głównie w pasie 0,2–0,6 wysokości kadru, poniżej ~0,7 praktycznie nie ma obiektów → ROI w górnych ~65–70% kadru.
- **Rozmiar a położenie:** obiekty wyżej w kadrze są większe (bliżej kamery), a w okolicy horyzontu (0,4–0,6) najmniejsze – można to wykorzystać jako prior rozmiaru kandydata.
- **Pora dnia** prawie nie zmienia rozmiarów ramek – trudność nocą wynika z wyglądu, a nie z geometrii.
- **Artefakty adnotacji:** 176 ramek zdegenerowanych (bok ≤ 1 px) i 35 ramek > 10% kadru – do odfiltrowania / ręcznego obejrzenia.
